# MIMIC-III — Leakage-Safe Hospital Mortality Baseline

**Portfolio working copy** based on the uploaded MIMIC-III notebook. This version uses an observed outcome (`hospital_expire_flag`), splits by patient **before fitting preprocessing**, and evaluates Logistic Regression and Random Forest on the **same untouched test set**.

**Important limitations:** This is a retrospective demonstration using admission-time demographic fields only. `hospital_expire_flag` is an in-hospital outcome; this is **not** a clinically validated risk model. The MIMIC-III demo dataset is small and may not support reliable subgroup comparisons. No raw patient-level data should be committed to a public GitHub repository. The original notebook's randomly generated targets, simulated bias, and diagnosis-count feature are intentionally not used as clinical predictors.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, RocCurveDisplay, average_precision_score,
                             balanced_accuracy_score)

RANDOM_STATE = 42
pd.set_option('display.max_columns', 40)


## 2. Load MIMIC-III demo files

Upload `PATIENTS.csv` and `ADMISSIONS.csv` to the Colab session, or mount Drive and set `DATA_DIR`. The files must be from the same authorized MIMIC-III release. Do **not** upload them to public GitHub.

In [ ]:



from google.colab import drive
from pathlib import Path

# Connect Google Drive
drive.mount('/content/drive')

# Location of MIMIC-III dataset
DATA_DIR = Path(
    '/content/drive/MyDrive/mimic-iii-clinical-database-demo-1.4'
)

# Verify the folder and required files
print("Dataset folder exists:", DATA_DIR.exists())

for filename in ['PATIENTS.csv', 'ADMISSIONS.csv']:
    path = DATA_DIR / filename
    print(filename, "FOUND" if path.is_file() else "MISSING")


In [ ]:

# Load the MIMIC-III datasets
patients = pd.read_csv(DATA_DIR / 'PATIENTS.csv')
admissions = pd.read_csv(DATA_DIR / 'ADMISSIONS.csv')

# Check dataset dimensions
print("Patients shape:", patients.shape)
print("Admissions shape:", admissions.shape)

# Verify essential columns
required_patients = {'subject_id', 'gender', 'dob'}
required_admissions = {
    'subject_id', 'hadm_id', 'admittime',
    'ethnicity', 'hospital_expire_flag'
}

assert required_patients.issubset(patients.columns), \
    "Missing required PATIENTS columns"

assert required_admissions.issubset(admissions.columns), \
    "Missing required ADMISSIONS columns"

print("All required columns found!")
print("Both datasets loaded successfully!")


## 3. Build one record per admission and validate the outcome

Predictors: sex, age at admission, and recorded ethnicity. Age is computed from DOB and admission time (rather than using the arbitrary year 2020). MIMIC-III shifts dates and may encode ages over 89 as approximately 300; those ages are capped at 90 for this teaching example. **Diagnosis counts are omitted** because complete discharge coding is not known at admission.

In [ ]:
patient_cols = ['subject_id', 'gender', 'dob']
admission_cols = ['subject_id', 'hadm_id', 'admittime', 'ethnicity', 'hospital_expire_flag']
for table, cols in [(patients, patient_cols), (admissions, admission_cols)]:
    absent = sorted(set(cols) - set(table.columns))
    if absent:
        raise ValueError(f'Required columns missing: {absent}')

p = patients[patient_cols].drop_duplicates('subject_id')
a = admissions[admission_cols].drop_duplicates('hadm_id')
df = a.merge(p, on='subject_id', how='inner', validate='many_to_one')

# Convert dates safely
df['admittime'] = pd.to_datetime(
    df['admittime'], errors='coerce'
)

df['dob'] = pd.to_datetime(
    df['dob'], errors='coerce'
)

# Calculate age using calendar years to avoid overflow
df['age'] = (
    df['admittime'].dt.year -
    df['dob'].dt.year
)

# Adjust if birthday has not yet occurred
birthday_not_reached = (
    (df['admittime'].dt.month < df['dob'].dt.month) |
    (
        (df['admittime'].dt.month == df['dob'].dt.month) &
        (df['admittime'].dt.day < df['dob'].dt.day)
    )
)

df['age'] = df['age'] - birthday_not_reached.astype(int)

# MIMIC-III age de-identification convention
df.loc[df['age'] > 89, 'age'] = 90

# Exclude invalid ages
df.loc[~df['age'].between(0, 120), 'age'] = np.nan

df.loc[df['age'] > 89, 'age'] = 90  # MIMIC-III de-identification convention
df.loc[~df['age'].between(0, 120), 'age'] = np.nan
df['hospital_expire_flag'] = pd.to_numeric(df['hospital_expire_flag'], errors='coerce')
df = df.dropna(subset=['subject_id', 'hadm_id', 'hospital_expire_flag']).copy()
if not df['hospital_expire_flag'].isin([0,1]).all():
    raise ValueError('hospital_expire_flag must contain only 0/1.')
df['hospital_expire_flag'] = df['hospital_expire_flag'].astype(int)
assert df['hadm_id'].is_unique
print('Admissions:', len(df), '| Unique patients:', df.subject_id.nunique())
print('Outcome distribution:')
print(df['hospital_expire_flag'].value_counts(dropna=False).to_string())
# Individual admission records are intentionally not displayed in this public notebook.


In [ ]:

print("Age summary:")
print(df['age'].describe())

print("\nMissing ages:", df['age'].isna().sum())

print("\nAge distribution:")
print(df['age'].value_counts().sort_index().tail(10))


In [ ]:

from sklearn.model_selection import GroupShuffleSplit

# Define the clinical prediction target
target_col = 'hospital_expire_flag'

# Split patients, not individual admission rows
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        y=df[target_col],
        groups=df['subject_id']
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

# Verify no patient appears in both datasets
train_patients = set(train_df['subject_id'])
test_patients = set(test_df['subject_id'])

overlap = train_patients.intersection(test_patients)

print("Training admissions:", len(train_df))
print("Testing admissions:", len(test_df))
print("Training patients:", len(train_patients))
print("Testing patients:", len(test_patients))
print("Overlapping patients:", len(overlap))

assert len(overlap) == 0, "Patient leakage detected!"
print("Patient-level split successful!")


In [ ]:

print("Training mortality rate:")
print(train_df['hospital_expire_flag'].mean())

print("\nTesting mortality rate:")
print(test_df['hospital_expire_flag'].mean())

print("\nTraining outcome counts:")
print(train_df['hospital_expire_flag'].value_counts())

print("\nTesting outcome counts:")
print(test_df['hospital_expire_flag'].value_counts())


In [ ]:

# Define input features and target
feature_cols = ['age', 'gender', 'ethnicity']
target_col = 'hospital_expire_flag'

X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

y_train = train_df[target_col].astype(int)
y_test = test_df[target_col].astype(int)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())


In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.impute import SimpleImputer

# Numerical and categorical features
numeric_features = ['age']
categorical_features = ['gender', 'ethnicity']

# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(
        strategy='most_frequent'
    )),
    ('encoder', OneHotEncoder(
        handle_unknown='ignore'
    ))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created!")


In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# Logistic Regression
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ))
])

# Random Forest
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

# Train models
lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

print("Logistic Regression trained successfully!")
print("Random Forest trained successfully!")


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score
)

import pandas as pd

models = {
    'Logistic Regression': lr_pipeline,
    'Random Forest': rf_pipeline
}

results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Balanced Accuracy': balanced_accuracy_score(
            y_test, y_pred
        ),
        'Precision': precision_score(
            y_test, y_pred, zero_division=0
        ),
        'Recall': recall_score(
            y_test, y_pred, zero_division=0
        ),
        'F1 Score': f1_score(
            y_test, y_pred, zero_division=0
        ),
        'ROC-AUC': (
            roc_auc_score(y_test, y_prob)
            if y_test.nunique() == 2
            else float('nan')
        )
    })

results_df = pd.DataFrame(results)

display(results_df.round(3))


In [ ]:

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_estimator(
        model,
        X_test,
        y_test,
        ax=ax,
        cmap='Blues',
        colorbar=False
    )
    ax.set_title(name)

plt.suptitle("MIMIC-III Mortality Prediction")
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=300,
            bbox_inches="tight")
plt.show()


In [ ]:

from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(8, 6))

for name, model in models.items():
    RocCurveDisplay.from_estimator(
        model,
        X_test,
        y_test,
        ax=ax,
        name=name
    )

ax.plot([0, 1], [0, 1], 'k--', label='Random baseline')
ax.set_title("ROC Curves — MIMIC-III Mortality Prediction")
ax.legend()

plt.tight_layout()
plt.savefig("roc_curves.png", dpi=300,
            bbox_inches="tight")
plt.show()


In [ ]:

results_df.to_csv("model_performance.csv", index=False)

print("Saved:")
print("- confusion_matrices.png")
print("- roc_curves.png")
print("- model_performance.csv")


## 4. Split by patient BEFORE preprocessing

No patient is allowed to appear in both training and testing. All imputation and encoding are fitted on training data only.

In [ ]:
FEATURES = ['gender', 'age', 'ethnicity']
TARGET = 'hospital_expire_flag'
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(df, df[TARGET], groups=df['subject_id']))
train_df, test_df = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()
overlap = set(train_df.subject_id) & set(test_df.subject_id)
assert not overlap, 'Patient leakage detected!'
X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]
print(f'Train admissions: {len(train_df)} | Test admissions: {len(test_df)}')
print(f'Patient overlap: {len(overlap)}')
print('Training outcome counts:', y_train.value_counts().to_dict())
print('Testing outcome counts:', y_test.value_counts().to_dict())
if y_train.nunique() < 2:
    raise ValueError('Training set has only one outcome class. A larger authorized dataset is required.')


## 5. Train leakage-safe pipelines

`class_weight="balanced"` is applied **during model fitting only** instead of duplicating patient records. Both models share the same split and preprocessing strategy.

In [ ]:
numeric_features = ['age']
categorical_features = ['gender', 'ethnicity']

def make_preprocessor():
    return ColumnTransformer(transformers=[
        ('numeric', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
        ]), numeric_features),
        ('categorical', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_features),
    ])

models = {
    'Logistic Regression': Pipeline([
        ('preprocess', make_preprocessor()),
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))
    ]),
    'Random Forest': Pipeline([
        ('preprocess', make_preprocessor()),
        ('model', RandomForestClassifier(n_estimators=200, min_samples_leaf=2,
                                         class_weight='balanced', random_state=RANDOM_STATE))
    ]),
}
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    print('Fitted:', name)


## 6. Evaluate on the untouched test set

ROC-AUC and average precision require both outcome classes in the test set. If the demo test set has only one class, they are reported as unavailable rather than fabricated.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
results = []
predictions = {}
for name, model in models.items():
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    predictions[name] = (pred, prob)
    row = {
        'model': name,
        'accuracy': accuracy_score(y_test, pred),
        'balanced_accuracy': balanced_accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, prob) if y_test.nunique()==2 else np.nan,
        'average_precision': average_precision_score(y_test, prob) if y_test.nunique()==2 else np.nan,
    }
    results.append(row)
    print(f'\n{name}\n', classification_report(y_test, pred, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=[0,1], cmap='Blues')
    plt.title(name + ' — test confusion matrix')
    plt.show()

results_df = pd.DataFrame(results).set_index('model')
display(results_df.round(3))
if y_test.nunique() == 2:
    fig, ax = plt.subplots(figsize=(7,5))
    for name, (pred, prob) in predictions.items():
        RocCurveDisplay.from_predictions(y_test, prob, name=name, ax=ax)
    ax.set_title('Test ROC curves')
    plt.show()
else:
    print('Warning: test data has only one class; ROC-AUC is undefined.')


## 7. Exploratory subgroup checks (not definitive fairness claims)

Small subgroup sizes and unequal outcome rates make fairness statistics unstable. Report denominators and do not treat missing groups as zero disparity. Gender and ethnicity are used for audit, not to establish clinical suitability.

In [ ]:
from sklearn.metrics import confusion_matrix

def subgroup_report(column, min_n=10):
    frame = test_df[[column, TARGET]].copy()
    frame['predicted'] = predictions['Logistic Regression'][0]
    frame[column] = frame[column].fillna('Missing').astype(str)
    rows = []
    for group, part in frame.groupby(column):
        y_true = part[TARGET].to_numpy()
        y_pred = part['predicted'].to_numpy()
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
        rows.append({
            'group': group, 'n': len(part), 'positives': int((y_true==1).sum()),
            'selection_rate': float(np.mean(y_pred)),
            'tpr': tp/(tp+fn) if tp+fn else np.nan,
            'fpr': fp/(fp+tn) if fp+tn else np.nan,
            'sufficient_n': len(part)>=min_n,
        })
    return pd.DataFrame(rows).sort_values('n', ascending=False)

for col in ['gender', 'ethnicity']:
    print('\nSubgroup audit:', col)
    display(subgroup_report(col))
print('CAUTION: No statistical confidence intervals or external validation were performed.')


## 8. Interpretation checklist

Before publishing results in the main README:

- [ ] Confirm the target is hospital mortality, not readmission or randomly assigned labels.
- [ ] Confirm the prediction time and that all predictors are available at that time.
- [ ] Check the number of positive cases in the held-out test set.
- [ ] Report actual test metrics, including poor results and uncertainty.
- [ ] Avoid conclusions about subgroup fairness when groups are very small.
- [ ] For stronger evidence, obtain an appropriately sized authorized dataset, add grouped cross-validation, and assess calibration and uncertainty.
- [ ] Keep restricted clinical data and credentials out of public GitHub.

**Notebook purpose:** reproducible educational baseline, not clinical deployment or validated medical decision support.

In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

# Create output folders
output_dir = Path('/content/portfolio_outputs')
figures_dir = output_dir / 'figures'
reports_dir = output_dir / 'reports'

figures_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

# Save evaluation results
results_df.to_csv(
    reports_dir / 'model_performance.csv',
    index=False
)

# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test,
        ax=ax, cmap='Blues', colorbar=False
    )
    ax.set_title(name)

plt.tight_layout()
plt.savefig(
    figures_dir / 'confusion_matrices.png',
    dpi=300, bbox_inches='tight'
)
plt.show()

# ROC curves
fig, ax = plt.subplots(figsize=(8, 6))

for name, model in models.items():
    RocCurveDisplay.from_estimator(
        model, X_test, y_test,
        ax=ax,
        name=name
    )

ax.plot([0, 1], [0, 1], 'k--', label='Chance')
ax.legend()
ax.set_title('MIMIC-III Mortality Prediction — ROC')

plt.tight_layout()
plt.savefig(
    figures_dir / 'roc_curves.png',
    dpi=300, bbox_inches='tight'
)
plt.show()

print("Portfolio outputs saved successfully!")
print("Location:", output_dir)
